In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib Tk

Raise $\Delta k$ if this takes too long to run.

In [3]:
dk=1;kmax=30;kmin=6
n=int(kmax/dk)*2+1
idk=dk*(np.array(np.where(np.zeros([n]*2)==0))-(kmax/dk)).astype(int)
order=lambda x: ((x[0]+kmax)/dk)*n+((x[1]+kmax)/dk)

In [4]:
%%time

k=[];p=[];q=[]

for i in range(len(idk.T)):
    tmp1=(np.linalg.norm(idk.T[i].reshape(2,1),axis=0)<=kmax) & (np.linalg.norm(idk,axis=0)<=kmax) & (np.linalg.norm(-idk-idk.T[i].reshape(2,1),axis=0)<=kmax)
    tmp2=(np.linalg.norm(idk.T[i].reshape(2,1),axis=0)>=kmin) & (np.linalg.norm(idk,axis=0)>=kmin) & (np.linalg.norm(-idk-idk.T[i].reshape(2,1),axis=0)>=kmin)
    tmp3=(order(idk.T[i])<order(idk)) & (order(idk)<order(-idk-idk.T[i].reshape(2,1)))
    if (tmp1&tmp2&tmp3).sum()!=0:
        k.append(np.array([idk.T[i]]*(tmp1&tmp2&tmp3).sum()))
        tmp=idk.T[tmp1&tmp2&tmp3]
        p.append(tmp)
        q.append(-tmp-idk.T[i])

k=np.row_stack(k).T;p=np.row_stack(p).T;q=np.row_stack(q).T

CPU times: user 505 ms, sys: 6.4 ms, total: 512 ms
Wall time: 511 ms


In [5]:
A=np.zeros([len(idk.T)]*2);G=np.zeros([len(idk.T)]*2)
A[order(k).astype(int),order(p).astype(int)]=1
A[order(k).astype(int),order(q).astype(int)]=1
A[order(p).astype(int),order(q).astype(int)]=1
A=A+A.T

G[order(k).astype(int),order(p).astype(int)]=(np.linalg.norm(k,axis=0)**2)*np.cross(q,p,axis=0)
G[order(k).astype(int),order(q).astype(int)]=(np.linalg.norm(k,axis=0)**2)*np.cross(p,q,axis=0)
G[order(p).astype(int),order(k).astype(int)]=(np.linalg.norm(p,axis=0)**2)*np.cross(q,k,axis=0)
G[order(p).astype(int),order(q).astype(int)]=(np.linalg.norm(p,axis=0)**2)*np.cross(k,q,axis=0)
G[order(q).astype(int),order(k).astype(int)]=(np.linalg.norm(q,axis=0)**2)*np.cross(p,k,axis=0)
G[order(q).astype(int),order(p).astype(int)]=(np.linalg.norm(q,axis=0)**2)*np.cross(k,p,axis=0)

In [6]:
# (A!=np.triu(A)).any()

### Old version (slower and more memory)

In [8]:
def get_triads(dk=1,kmax=2,kmin=0):
    if int(kmax/dk)!=kmax/dk:
        print('kmax must divide evenly by dk')
        return
        
    n=int(kmax/dk)*2+1
    A=np.zeros([n]*4)
    # G=np.zeros(A.shape)
    idx=dk*(np.array(np.where(A==0))-(kmax/dk)).astype(int)
    
    tmp1=((-idx[0]-idx[2])**2+(-idx[1]-idx[3])**2<=kmax**2) & (idx[0]**2+idx[1]**2<=kmax**2) & (idx[2]**2+idx[3]**2<=kmax**2) & ~((idx[0]==idx[2])&(idx[1]==idx[3]))
    tmp2=((-idx[0]-idx[2])**2+(-idx[1]-idx[3])**2>=kmin**2) & (idx[0]**2+idx[1]**2>=kmin**2) & (idx[2]**2+idx[3]**2>=kmin**2)
    triads=idx[:,tmp1&tmp2]
    
    k=triads[[0,1]]
    p=triads[[2,3]]
    q=-triads[[0,1]]-triads[[2,3]]
    
    # grid=np.zeros([n]*2)
    # grid=np.array(np.where(grid==0))-(kmax*dk)
    # order=lambda x: np.argmax((grid[:,:,None]==x[:,None,:]).all(axis=0),axis=0)
    order=lambda x: ((x[0]+kmax/dk)*n+(x[1]+kmax/dk))
    
    idx=(order(k)<order(p))&(order(p)<order(q))
    k=k[:,idx];p=p[:,idx];q=q[:,idx]
    # for i in range(len(k.T)):
    #     print(f'k: {k[:,i]}, p: {p[:,i]}, q: {q[:,i]}')

    # A[k[0],k[1],p[0],p[1]]=1
    # G[k[0],k[1],p[0],p[1]]=(np.linalg.norm(k,axis=0)**2)*np.cross(q,p,axis=0)

    # return A.astype(int),G
    return k,p,q


In [9]:
# %%time
# A,G=get_triads(dk=0.5,kmax=30,kmin=6)
# # k,p,q=get_triads(dk=1,kmax=30,kmin=6)

In [10]:
%%time
k1,p1,q1=get_triads(dk=dk,kmax=kmax,kmin=kmin)

CPU times: user 526 ms, sys: 330 ms, total: 856 ms
Wall time: 855 ms


In [11]:
print((k!=k1).any())
print((p!=p1).any())
print((q!=q1).any())

False
False
False
